# Assignment 6: Spectral Inferences and the Subsequent Memory Effect
This assignment scales the spectral analysis of Assignment 5 up to a 20-subject intracranial cohort and asks what you can infer from it: which frequencies and which brain regions carry a subsequent memory effect (SME), how the answer depends on how you normalise power, and how it depends on how you reference the electrodes. It uses intracranial EEG from the Delayed Free Recall of Word Lists study (FR1, https://openneuro.org/datasets/ds004789/versions/3.1.0), reported in Solomon et al. (2018).


## 📥 Saving your answers for grading

This assignment is **auto-graded**. After each question there is a **grader cell**
that saves the data you plotted or computed into an `answers/Module_14/` folder so
it can be compared against the reference answers.

**For each question:**
1. The question tells you which result(s) to produce and the expected data
   structure/format. Do your analysis and bind each result to a variable.
2. In the grader cell, replace the placeholder variable with **your** variable name.
3. Run the grader cell — it calls `save_answer(...)` and writes your answer.

Your **plots are saved too** — make sure each plotting cell calls `plt.show()` so the
figure can be captured for the grade report.

Make sure every grader cell runs without error before you submit. You don't need to
change anything else.


In [ ]:
# grader setup — enables saving the figures you plot (run once, early)
try:
    from grader.answer_io import enable_figure_capture
    enable_figure_capture()
except Exception as _e:
    print("grader: figure capture not enabled:", _e)


In [ ]:
# imports
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats
import mne
import ptsa
from ptsa.data.filters import morlet
from ptsa.data.filters import ButterworthFilter
sys.path.insert(0, 'dependencies/bidsreader')
from bidsreader import CMLBIDSReader as BIDSReader
from bidsreader import mne_epochs_to_ptsa
sys.path.insert(0, '.')
from cml_data import get_bids_root, INTRAC_SUBS, INTRAC_SUBS_SMALL


## Assignment Overview

Much like the previous assignment, in this project you will analyze multi-session free recall experiments to determine the spectral biomarkers of successful memory encoding. To determine differences in brain activity that predict whether a studied item will be subsequently remembered (recalled) you will compare two classes of events: word encodings that led to later recall and those that did not. The data were recorded from patients with epilepsy undergoing a neurosurgical evaluation that required implantation of depth and/or grid electrodes (Solomon et al., 2018).

A key step in processing intracranial data is using CT and MRI scans to localize each electrode and match it to a standard atlas so that every contact carries a brain-region label. That step has been done for you: the labels are in the electrode table that `reader.load_combined_channels(acquisition="bipolar")` returns alongside every channel (columns such as `ind.region_ch1`, `ind.region_ch2`, `wb.region_ch1`, ...).

You will analyze the cohort `INTRAC_SUBS` from `cml_data.py` (the same 20 subjects as Module 10). Each was chosen because they have at least three sessions of FR1 data, recall performance of at least 30%, and electrodes in temporal cortex, frontal cortex and hippocampus. For every session of every subject use the processing steps you implemented in Module 10 (they should already live in your `SpectralHelpers.py`):
* Load the **bipolar** EEG for all WORD events with a 1000 ms buffer on each side of the 1600 ms encoding window.
* Apply a Butterworth notch filter around 60 Hz (freqs = [58, 62]).
* Compute power at the 16 frequencies below with a Morlet wavelet with wavenumber (`width`) 6 for each encoding event, remove the buffer, and average power over the 1600 ms encoding window.
* Save each session's (events x channels x frequencies) array to disk as soon as it is computed. A crash on session 47 of 70 should cost you one session, not an afternoon: check for the saved file first and skip sessions you have already done.
* Be robust to exceptions. This data was acquired from patients across many years, and small things go wrong (a missing file, a session with no recalled words). Catch the exception, **log** the subject/session and the error message, and discard that session, but keep going.
* In some cases you may notice artifacts that manifest as power values of exactly zero, which break the log transform. Exclude such events from all analyses.

The whole cohort is roughly 70 sessions and 30-50 GB of EEG, so develop and test your code on `INTRAC_SUBS_SMALL` (the first three subjects) before running it over `INTRAC_SUBS`. Module 12 shows how to run the per-session function in parallel; for now a plain loop that caches to disk is fine.

All logarithms referenced in this assignment are base-10 logarithms (np.log10).


In [ ]:
# the intracranial cohort; develop on INTRAC_SUBS_SMALL, run on INTRAC_SUBS
intrac_subs = INTRAC_SUBS
print(len(intrac_subs), "subjects:", intrac_subs)

freqs = np.unique(np.round(np.logspace(np.log10(1), np.log10(300), 17)))
print("frequencies (Hz):", freqs)


## Question 1: Between-subject power spectra
Repeat the Module 10 analysis for every subject in `intrac_subs`, averaging over all bipolar channels, and make statistical inferences about spectral biomarkers of successful memory encoding.

1) Compute the average power spectrum for each subject (separately for recalled and non-recalled items) and then take the log of that spectrum. Averaging these log-spectra across subjects, graph the average log-power spectra with 95% confidence bands (transparent light red and light blue shading, or your favorite clearly labeled color scheme) on a semi-log-x plot.
2) To get a sense for the effect of the log-transform and the semi-log plot, also produce the plot without the log-transformation of the subject-level powers (semi-log-x, linear y) and ...
3) ... also produce the plot without either the log transformation or the semi-log axis.

In other words, you're manipulating your x and y axes. You should have standard PSD values, log PSD values, and both standard and logged frequency axes. The goal is to see first hand how the log transform changes what you can read off the data.

4) Plot the mean of the differences between the log spectra (recalled - not recalled) computed separately for each subject (and then averaged across subjects) on a semi-log-x plot and place a 95% confidence band on the mean difference score.

5) What inferences can you now make from these results? Which frequency ranges show a subsequent memory effect, and in which direction?

<!-- grader-note -->
> **📥 For grading, produce and save:** `freqs` (array (16,) — x-axis frequencies (16 log-spaced, 1..300 Hz), shared by all Q1/Q2 spectra plots) → Q1.1; `rec_log_mean` (array (16,) — across-subject mean of per-subject log10 power spectra, recalled (part 1)) → Q1.1; `nrec_log_mean` (array (16,) — across-subject mean of per-subject log10 power spectra, non-recalled (part 1)) → Q1.1; `rec_avg` (array (16,) — across-subject mean power (no log-transform), recalled (part 2, semilog-x)) → Q1.2; `nrec_avg` (array (16,) — across-subject mean power (no log-transform), non-recalled (part 2, semilog-x)) → Q1.2; `rec_avg` (array (16,) — same recalled mean power as Q1.2, plotted on linear axes (part 3)) → Q1.3; `nrec_avg` (array (16,) — same non-recalled mean power as Q1.2, plotted on linear axes (part 3)) → Q1.3; `log_diff_mean` (array (16,) — across-subject mean of per-subject (recalled - non-recalled) log-power differences (part 4)) → Q1.4. Bind each to a variable, then run the grader cell(s) below.
<!-- /grader-note -->

In [ ]:
# Question 1.1
### YOUR CODE HERE


In [ ]:
# ── grader cell (Question 1.1) ── saves your answer(s); edit the variable name ──
from grader.answer_io import save_answer
save_answer("Q1.1_freqs", freqs, module=14, question="1.1", fig="last")   # ← replace `freqs` with your variable
save_answer("Q1.1_rec_log_mean", rec_log_mean, module=14, question="1.1")   # ← replace `rec_log_mean` with your variable
save_answer("Q1.1_nrec_log_mean", nrec_log_mean, module=14, question="1.1")   # ← replace `nrec_log_mean` with your variable

In [ ]:
# Question 1.2
### YOUR CODE HERE


In [ ]:
# ── grader cell (Question 1.2) ── saves your answer(s); edit the variable name ──
from grader.answer_io import save_answer
save_answer("Q1.2_rec_avg", rec_avg, module=14, question="1.2", fig="last")   # ← replace `rec_avg` with your variable
save_answer("Q1.2_nrec_avg", nrec_avg, module=14, question="1.2")   # ← replace `nrec_avg` with your variable

In [ ]:
# Question 1.3
### YOUR CODE HERE


In [ ]:
# ── grader cell (Question 1.3) ── saves your answer(s); edit the variable name ──
from grader.answer_io import save_answer
save_answer("Q1.3_rec_avg", rec_avg, module=14, question="1.3", fig="last")   # ← replace `rec_avg` with your variable
save_answer("Q1.3_nrec_avg", nrec_avg, module=14, question="1.3")   # ← replace `nrec_avg` with your variable

In [ ]:
# Question 1.4
### YOUR CODE HERE


In [ ]:
# ── grader cell (Question 1.4) ── saves your answer(s); edit the variable name ──
from grader.answer_io import save_answer
save_answer("Q1.4_log_diff_mean", log_diff_mean, module=14, question="1.4", fig="last")   # ← replace `log_diff_mean` with your variable

Question 1.5

**YOUR ANSWER HERE**


## Question 2: Normalisation
In this problem, your goal is to assess the effects of two important processing steps on the analysis in the previous problem. Two common normalization procedures used in the analysis of brain signals are the log-transform, which attenuates extremely large values, and the z-transform, which normalizes power values against some baseline distribution.

The log-transform can be applied at different points in the analysis stream: immediately after computing power values (sample by sample), after averaging power over each encoding interval, or after computing the z-transform. Similarly, the z-transform can be applied to power (or log-power) using many choices of the reference "distribution". If our goal is to normalize to the distribution of power values across a given session, how do you define the distribution over which to estimate the mean and standard deviation? Here you will reproduce the between-subject (recalled - non-recalled) comparison of Question 1 under each choice.

1) Take the logarithm of power **before** averaging across time (i.e. log10 each power sample, then average over the encoding window), and plot the between-subject mean log-power spectra for recalled and non-recalled words with confidence bands.
2) Z-score the (time-averaged) power across the events within each session, separately for every channel and frequency, and plot the between-subject mean z-scored spectra for recalled and non-recalled words with confidence bands.
3) Complete a third transformation of your choice, and plot it the same way.
4) Describe your motivation in considering the transform in part 3. Assess the effects of the different choices (parts 1, 2, 3) on the apparent subsequent memory effect.

<!-- grader-note -->
> **📥 For grading, produce and save:** `rec_logpow` (array (16,) — across-subject mean spectrum of log10 power taken sample-by-sample before time-averaging, recalled) → Q2.1; `nrec_logpow` (array (16,) — across-subject mean spectrum of log10 power taken before time-averaging, non-recalled) → Q2.1; `rec_z_mean` (array (16,) — across-subject mean of within-session z-scored power, recalled) → Q2.2; `nrec_z_mean` (array (16,) — across-subject mean of within-session z-scored power, non-recalled) → Q2.2. Bind each to a variable, then run the grader cell(s) below.
<!-- /grader-note -->

In [ ]:
# Question 2.1
### YOUR CODE HERE


In [ ]:
# ── grader cell (Question 2.1) ── saves your answer(s); edit the variable name ──
from grader.answer_io import save_answer
save_answer("Q2.1_rec_logpow", rec_logpow, module=14, question="2.1", fig="last")   # ← replace `rec_logpow` with your variable
save_answer("Q2.1_nrec_logpow", nrec_logpow, module=14, question="2.1")   # ← replace `nrec_logpow` with your variable

In [ ]:
# Question 2.2
### YOUR CODE HERE


In [ ]:
# ── grader cell (Question 2.2) ── saves your answer(s); edit the variable name ──
from grader.answer_io import save_answer
save_answer("Q2.2_rec_z_mean", rec_z_mean, module=14, question="2.2", fig="last")   # ← replace `rec_z_mean` with your variable
save_answer("Q2.2_nrec_z_mean", nrec_z_mean, module=14, question="2.2")   # ← replace `nrec_z_mean` with your variable

In [ ]:
# Question 2.3
### YOUR CODE HERE


Question 2.4

**YOUR ANSWER HERE**


## Question 3: Electrodes by brain region
All of the preceding analyses pooled every channel of every subject. Now we ask *where* in the brain the effect lives. Assign each bipolar pair to one of three regions from its localisation labels:

| region | look in | label contains |
|---|---|---|
| temporal cortex | `ind.region_ch1`, `ind.region_ch2` | `temporal` or `bankssts` |
| frontal cortex | `ind.region_ch1`, `ind.region_ch2` | `frontal` |
| hippocampus | `wb.region_ch1`, `wb.region_ch2` (also `stein.region_*`) | `hippocampus` (but **not** `parahippocampal`, which is a temporal gyrus) |

A pair belongs to a region if **either** of its two contacts matches, and a pair can belong to more than one region. Hippocampal contacts usually have an empty `ind.region` (the cortical atlas has no label for them), which is why they are looked up in the whole-brain atlas column instead.

1) Create a table with one row per subject and the number of bipolar pairs in each of the three regions (columns `subject, temporal, frontal, hippocampus`). Use each subject's first FR1 session; montages rarely change between sessions, but note any subject where they do.

<!-- grader-note -->
> **📥 For grading, produce and save:** `region_counts` (dataframe: subject, temporal, frontal, hippocampus — one row per cohort subject: number of bipolar pairs in temporal cortex, frontal cortex and hippocampus (first FR1 session)) → Q3.1. Bind each to a variable, then run the grader cell(s) below.
<!-- /grader-note -->

In [ ]:
# Question 3.1
### YOUR CODE HERE


In [ ]:
# ── grader cell (Question 3.1) ── saves your answer(s); edit the variable name ──
from grader.answer_io import save_answer
save_answer("Q3.1_region_counts", region_counts, module=14, question="3.1")   # ← replace `region_counts` with your variable

## Question 4: Regional subsequent memory effects
Assess the subsequent memory effect at a particular frequency across locations in the brain.
* Instead of topographic maps (which do not exist for intracranial electrodes), analyze the data for channels pooled within the **temporal** and **frontal** regions from Question 3.
* Conduct these analyses on only those subjects who have at least one bipolar pair in each of the temporal and frontal regions.
* Log-transform the power immediately after extracting it, then z-transform within each session across events (the two transforms from Question 2, in that order).
* Because subjects will have different numbers of electrodes in each region, you need to decide how to combine data across electrodes.
* Complete the analysis at both **10 Hz** and **147 Hz**.

1) Think of a reasonable way to combine data across electrodes and try it. Plot the SME (recalled vs. non-recalled mean z-power with SEM error bars) for the two regions at each frequency.
2) Think of another reasonable way to combine data across electrodes and try it. One defensible choice, and the one the reference answer uses: average z-power over all of a subject's electrodes in a region so each subject contributes one number per region and condition, then compute the mean and SEM across subjects. Plot it the same way. Then test the SME statistically: for each region x frequency (4 tests) run a paired t-test of recalled vs. non-recalled subject means, correct the four p-values with Benjamini-Hochberg FDR at q < 0.05, and mark the significant region/frequency combinations on the plot.
3) Report the approaches that you used, as well as the strengths and weaknesses of each. There may not be a "right" answer here; use your best judgement and argue for the method you think is best. Where is the SME, and does the answer depend on the frequency?

<!-- grader-note -->
> **📥 For grading, produce and save:** `rec_means_10hz` (array (2,) — recalled mean of subject-level regional z-power [temporal, frontal] at 10 Hz, bipolar reference (errorbar y)) → Q4.2; `rec_sems_10hz` (array (2,) — recalled SEM across subjects [temporal, frontal] at 10 Hz, bipolar reference (errorbar yerr)) → Q4.2; `nrec_means_10hz` (array (2,) — not-recalled mean of subject-level regional z-power [temporal, frontal] at 10 Hz, bipolar reference (errorbar y)) → Q4.2; `nrec_sems_10hz` (array (2,) — not-recalled SEM across subjects [temporal, frontal] at 10 Hz, bipolar reference (errorbar yerr)) → Q4.2; `rec_means_147hz` (array (2,) — recalled mean of subject-level regional z-power [temporal, frontal] at 147 Hz, bipolar reference (errorbar y)) → Q4.2; `rec_sems_147hz` (array (2,) — recalled SEM across subjects [temporal, frontal] at 147 Hz, bipolar reference (errorbar yerr)) → Q4.2; `nrec_means_147hz` (array (2,) — not-recalled mean of subject-level regional z-power [temporal, frontal] at 147 Hz, bipolar reference (errorbar y)) → Q4.2; `nrec_sems_147hz` (array (2,) — not-recalled SEM across subjects [temporal, frontal] at 147 Hz, bipolar reference (errorbar yerr)) → Q4.2; `sme_fdr_sig` (array (4,) — boolean: which of [temporal@10Hz, frontal@10Hz, temporal@147Hz, frontal@147Hz] survive BH-FDR at q<0.05 (paired t-test, recalled vs non-recalled subject means)) → Q4.2. Bind each to a variable, then run the grader cell(s) below.
<!-- /grader-note -->

In [ ]:
# Question 4.1
### YOUR CODE HERE


In [ ]:
# Question 4.2
### YOUR CODE HERE


In [ ]:
# ── grader cell (Question 4.2) ── saves your answer(s); edit the variable name ──
from grader.answer_io import save_answer
save_answer("Q4.2_rec_means_10hz", rec_means_10hz, module=14, question="4.2", fig="last")   # ← replace `rec_means_10hz` with your variable
save_answer("Q4.2_rec_sems_10hz", rec_sems_10hz, module=14, question="4.2")   # ← replace `rec_sems_10hz` with your variable
save_answer("Q4.2_nrec_means_10hz", nrec_means_10hz, module=14, question="4.2")   # ← replace `nrec_means_10hz` with your variable
save_answer("Q4.2_nrec_sems_10hz", nrec_sems_10hz, module=14, question="4.2")   # ← replace `nrec_sems_10hz` with your variable
save_answer("Q4.2_rec_means_147hz", rec_means_147hz, module=14, question="4.2")   # ← replace `rec_means_147hz` with your variable
save_answer("Q4.2_rec_sems_147hz", rec_sems_147hz, module=14, question="4.2")   # ← replace `rec_sems_147hz` with your variable
save_answer("Q4.2_nrec_means_147hz", nrec_means_147hz, module=14, question="4.2")   # ← replace `nrec_means_147hz` with your variable
save_answer("Q4.2_nrec_sems_147hz", nrec_sems_147hz, module=14, question="4.2")   # ← replace `nrec_sems_147hz` with your variable
save_answer("Q4.2_sme_fdr_sig", sme_fdr_sig, module=14, question="4.2")   # ← replace `sme_fdr_sig` with your variable

Question 4.3

**YOUR ANSWER HERE**


## Question 5: Referencing
For the preceding analysis, you used bipolar referencing. As with scalp EEG you can instead use an average reference. With intracranial EEG, however, the average reference is not always well defined because each subject has their own idiosyncratic arrangement of electrodes, with some regions, such as medial temporal lobe and lateral temporal cortex, being systematically over-sampled for the clinical treatment of epilepsy. For this problem:
* Load the **monopolar** contacts (`acquisition="monopolar"`) instead of the bipolar pairs, and assign contacts to regions with the same rules as Question 3 (now using `ind.region` / `wb.region` directly).
* Create an average reference by first averaging the voltage across contacts within each of the three regions, then averaging those regional averages, using only regions with at least 5 contacts. Subtract this reference from every contact before filtering and computing power. This ensures that each region is equally weighted within the overall average.

1) Recompute the Question 4.2 analysis (subject-level regional averages, 10 Hz and 147 Hz, means and SEMs) using this reference.
2) Compare your results to the bipolar ones and comment on the strengths and weaknesses of each referencing scheme. Do the differences depend on the frequency being examined?

<!-- grader-note -->
> **📥 For grading, produce and save:** `rec_means_10hz` (array (2,) — recalled mean of subject-level regional z-power [temporal, frontal] at 10 Hz, region-weighted average reference) → Q5.1; `rec_sems_10hz` (array (2,) — recalled SEM across subjects [temporal, frontal] at 10 Hz, region-weighted average reference) → Q5.1; `nrec_means_10hz` (array (2,) — not-recalled mean of subject-level regional z-power [temporal, frontal] at 10 Hz, region-weighted average reference) → Q5.1; `nrec_sems_10hz` (array (2,) — not-recalled SEM across subjects [temporal, frontal] at 10 Hz, region-weighted average reference) → Q5.1; `rec_means_147hz` (array (2,) — recalled mean of subject-level regional z-power [temporal, frontal] at 147 Hz, region-weighted average reference) → Q5.1; `rec_sems_147hz` (array (2,) — recalled SEM across subjects [temporal, frontal] at 147 Hz, region-weighted average reference) → Q5.1; `nrec_means_147hz` (array (2,) — not-recalled mean of subject-level regional z-power [temporal, frontal] at 147 Hz, region-weighted average reference) → Q5.1; `nrec_sems_147hz` (array (2,) — not-recalled SEM across subjects [temporal, frontal] at 147 Hz, region-weighted average reference) → Q5.1. Bind each to a variable, then run the grader cell(s) below.
<!-- /grader-note -->

In [ ]:
# Question 5.1
### YOUR CODE HERE


In [ ]:
# ── grader cell (Question 5.1) ── saves your answer(s); edit the variable name ──
from grader.answer_io import save_answer
save_answer("Q5.1_rec_means_10hz", rec_means_10hz, module=14, question="5.1", fig="last")   # ← replace `rec_means_10hz` with your variable
save_answer("Q5.1_rec_sems_10hz", rec_sems_10hz, module=14, question="5.1")   # ← replace `rec_sems_10hz` with your variable
save_answer("Q5.1_nrec_means_10hz", nrec_means_10hz, module=14, question="5.1")   # ← replace `nrec_means_10hz` with your variable
save_answer("Q5.1_nrec_sems_10hz", nrec_sems_10hz, module=14, question="5.1")   # ← replace `nrec_sems_10hz` with your variable
save_answer("Q5.1_rec_means_147hz", rec_means_147hz, module=14, question="5.1")   # ← replace `rec_means_147hz` with your variable
save_answer("Q5.1_rec_sems_147hz", rec_sems_147hz, module=14, question="5.1")   # ← replace `rec_sems_147hz` with your variable
save_answer("Q5.1_nrec_means_147hz", nrec_means_147hz, module=14, question="5.1")   # ← replace `nrec_means_147hz` with your variable
save_answer("Q5.1_nrec_sems_147hz", nrec_sems_147hz, module=14, question="5.1")   # ← replace `nrec_sems_147hz` with your variable

Question 5.2

**YOUR ANSWER HERE**
